In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

BASE        = next(p for p in [Path().resolve()] + list(Path().resolve().parents) if (p / "DECISIONS.md").exists())
HORAS_IN    = BASE / "data/processed/horas_por_setor_2023.csv"
PIB_IN      = BASE / "data/processed/pib_setorial_2012_2023.csv"
BASE_OUT    = BASE / "data/processed/base_analitica_setorial_2023.csv"
TAB_OUT     = BASE / "outputs/tables/impacto_setorial_hipotese_nula.csv"
DIR_FIGS    = BASE / "outputs/figures"

plt.rcParams.update({
    "figure.dpi": 150,
    "font.family": "sans-serif",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
})
print("Imports OK")

In [ ]:
df_horas = pd.read_csv(HORAS_IN, index_col=0)
df_pib   = pd.read_csv(PIB_IN)

pib_2023 = df_pib[df_pib["ano"] == 2023].squeeze()
pib_total_rs = pib_2023["pib_total"] * 1e6  

print(f"PIB total 2023: R$ {pib_total_rs/1e12:.2f} trilhões")
print(f"\nSetores disponíveis nas horas:")
print(df_horas.index.tolist())

In [ ]:
correspondencia = [
    
    ("Agropecuária",           "agropecuaria",          True,
     "Correspondência direta"),

    ("Indústria geral",        "ind_transformacao",      True,
     "CNT separa subsetores; ind_transformacao é a maior parcela (~60% do VA industrial)"),

    ("Construção",             "construcao",             True,
     "Correspondência direta"),

    ("Comércio e rep.",        "comercio",               True,
     "Correspondência direta"),

    ("Transp. e armaz.",       "transporte",             True,
     "Correspondência direta"),

    ("Inf. e serv. prof.",     "info_comunicacao",       True,
     "Proxy parcial: CNT separa info e serv. profissionais; usamos info_comunicacao"),

    ("Adm. públ. e educação",  "adm_publica_saude_educ", True,
     "CNT agrega adm. pública + saúde + educação públicas. Aceito para fins de proxy."),

    ("Alojamento e alim.",     "outros_servicos",        False,
     "CNT não separa alojamento; outros_servicos é agregado muito amplo. Excluído."),

    ("Saúde",                  "adm_publica_saude_educ", False,
     "Saúde pública agregada com adm. pública na CNT. VAB não separável. Excluído."),

    ("Serv. domésticos",       "outros_servicos",        False,
     "VAB formal muito baixo; proxy não representativa. Excluído."),
]

df_corr = pd.DataFrame(correspondencia,
    columns=["setor_pnad", "col_cnt", "incluir_simulacao", "justificativa"])

print("── Tabela de correspondência ────────────────────────────")
print(df_corr[["setor_pnad", "col_cnt", "incluir_simulacao"]].to_string(index=False))

In [ ]:
df_horas = df_horas.rename(columns={
    "pct_acima_44h": "pct_acima_44",
    "pct_40h_exatas": "pct_40h_exatas",
})
print("Colunas disponíveis:")
print(df_horas.columns.tolist())

In [ ]:
SEMANAS_ANO = 52
HORAS_ATUAL   = 44.0
HORAS_PROPOSTA = 40.0

registros = []

for _, row in df_corr.iterrows():
    setor  = row["setor_pnad"]
    col    = row["col_cnt"]
    incluir = row["incluir_simulacao"]

    if setor not in df_horas.index:
        print(f"Setor não encontrado nas horas: {setor}")
        continue

    n_ocupados  = df_horas.loc[setor, "n"]
    h_media     = df_horas.loc[setor, "media"]
    h_mediana   = df_horas.loc[setor, "mediana"]
    pct_acima44 = df_horas.loc[setor, "pct_acima_44"]
    pct_40_44   = df_horas.loc[setor, "pct_40_44"]

    vab_rs = pib_2023[col] * 1e6  

    total_horas_ano = n_ocupados * h_media * SEMANAS_ANO

    produt_hora = vab_rs / total_horas_ano

    # Só quem trabalha mais de 40h precisa reduzir esse é o universo afetado.
    # Limitamos a faixa em 44h porque acima disso já é hora extra, outra história.
    # A redução média de 42 -> 40h vem da média simples da faixa só vira parâmetro depois.

    h_media_afetados = 42.0  
    reducao_relativa_afetados = (HORAS_PROPOSTA - h_media_afetados) / h_media_afetados  # -4,76%

    parcela_afetada = pct_40_44 / 100.0

    delta_horas_setor = parcela_afetada * reducao_relativa_afetados

    delta_vab_rs = vab_rs * delta_horas_setor

    registros.append({
        "setor":                setor,
        "col_cnt":              col,
        "incluir_simulacao":    incluir,
        "n_ocupados":           int(n_ocupados),
        "h_media":              round(h_media, 2),
        "h_mediana":            round(h_mediana, 1),
        "pct_41_44h":           round(pct_40_44, 1),
        "pct_acima_44h":        round(pct_acima44, 1),
        "vab_r$_bi":            round(vab_rs / 1e9, 2),
        "total_horas_ano_M":    round(total_horas_ano / 1e6, 2),
        "produt_r$_hora":       round(produt_hora, 2),
        "parcela_afetada":      round(parcela_afetada, 4),
        "delta_horas_setor":    round(delta_horas_setor, 6),
        "delta_vab_r$_bi":      round(delta_vab_rs / 1e9, 3),
        "delta_vab_pct_setor":  round(delta_horas_setor * 100, 3),
    })

df_base = pd.DataFrame(registros)

print("\n── Base analítica setorial 2023 ─────────────────────────")
cols_print = ["setor", "h_media", "pct_41_44h", "vab_r$_bi",
              "produt_r$_hora", "delta_vab_r$_bi", "delta_vab_pct_setor"]
print(df_base[cols_print].to_string(index=False))

In [ ]:
df_sim = df_base[df_base["incluir_simulacao"] == True].copy()

delta_vab_total_rs  = df_sim["delta_vab_r$_bi"].sum() * 1e9
delta_pib_pct       = (delta_vab_total_rs / pib_total_rs) * 100
delta_pib_bi        = delta_vab_total_rs / 1e9

print("\n══ RESULTADO — HIPÓTESE NULA (sem ganho de produtividade) ══")
print(f"Setores incluídos na simulação: {df_sim['setor'].tolist()}")
print(f"ΔVAB total (setores incluídos): R$ {delta_pib_bi:.2f} bi")
print(f"ΔPIB estimado (% do PIB total): {delta_pib_pct:.3f}%")
print(f"\nReferência CNI:                 -0,70%")
print(f"Nosso modelo (hipótese nula):   {delta_pib_pct:.3f}%")
print(f"\nNota: diferença esperada — metodologias distintas.")
print(f"Nosso modelo usa apenas trabalhadores na faixa 41-44h.")
print(f"CNI pode usar elasticidades setoriais diferentes.")

In [ ]:
df_base.to_csv(BASE_OUT, index=False)
df_sim[cols_print + ["delta_vab_r$_bi", "delta_vab_pct_setor"]].to_csv(TAB_OUT, index=False)

print(f"\nBase analítica salva: {BASE_OUT}")
print(f"Tabela de impacto salva: {TAB_OUT}")

In [ ]:
df_g = df_sim.sort_values("delta_vab_r$_bi", ascending=True)

fig, ax = plt.subplots(figsize=(11, 6))

cores = ["#E84545" if v < -1 else "#F5A623" if v < -0.5 else "#2C6FBF"
         for v in df_g["delta_vab_r$_bi"]]

bars = ax.barh(df_g["setor"], df_g["delta_vab_r$_bi"], color=cores, height=0.6)
ax.axvline(0, color="black", linewidth=0.8)

for bar, (_, row) in zip(bars, df_g.iterrows()):
    val = row["delta_vab_r$_bi"]
    ax.text(
        val - 0.05 if val < 0 else val + 0.05,
        bar.get_y() + bar.get_height() / 2,
        f"R$ {val:.2f} bi  ({row['delta_vab_pct_setor']:.2f}%)",
        va="center", ha="right" if val < 0 else "left", fontsize=9
    )

ax.set_xlabel("Variação estimada no VAB (R$ bilhões)", fontsize=11)
ax.set_title(
    "Impacto setorial da redução de jornada (44h→40h) — hipótese nula\n"
    "Sem ganho de produtividade · Trabalhadores na faixa 41–44h · Brasil 2023",
    fontsize=12, loc="left"
)

fig.text(
    0.01, -0.03,
    f"ΔPIB agregado estimado (setores incluídos): {delta_pib_pct:.3f}% "
    f"(R$ {delta_pib_bi:.1f} bi) · "
    "Referência CNI: -0,70% · Metodologias distintas — ver DECISIONS.md",
    fontsize=7.5, color="gray", transform=fig.transFigure
)

plt.tight_layout()
plt.savefig(DIR_FIGS / "11_impacto_setorial_hipotese_nula.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gráfico 11 salvo.")

In [ ]:
import pandas as pd
from pathlib import Path

BASE     = next(p for p in [Path().resolve()] + list(Path().resolve().parents) if (p / "DECISIONS.md").exists())
df_horas = pd.read_csv(BASE / "data/processed/horas_por_setor_2023.csv", index_col=0)

print("Colunas disponíveis:")
print(df_horas.columns.tolist())
print()
print("Valores completos:")
print(df_horas[["n", "media", "mediana", "pct_abaixo_40", "pct_40_44", "pct_acima_44"]].to_string())

In [ ]:
import pandas as pd
from pathlib import Path

BASE     = next(p for p in [Path().resolve()] + list(Path().resolve().parents) if (p / "DECISIONS.md").exists())

df_horas = pd.read_csv(
    BASE / "data/processed/horas_por_setor_2023_corrigido.csv",
    index_col=0
)

df_horas = df_horas.rename(columns={"pct_41_44h": "pct_40_44"})

print("Arquivo corrigido carregado.")
print(df_horas[["n", "media", "pct_40_44", "pct_acima_44h"]].to_string())